# SWYNEX – Exploratory Data Analysis (Online Retail)

**Objective:** Explore the cleaned Online Retail dataset from Task 1, calculate descriptive statistics, identify trends/patterns/anomalies, and communicate at least five useful insights.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn.

**Important metric definition:** Sales revenue charts use rows where `LineTotal > 0` and `IsCancellation == False`. This keeps positive-value sales lines separate from cancellations, returns/negative adjustments, and zero-value rows. It is a descriptive analysis, not audited accounting revenue.

**Period caveat:** The source data ends on 9 December 2011, so December 2011 is incomplete.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path("../data/online_retail_cleaned.csv")
df = pd.read_csv(DATA_PATH)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")
df.head()


## 1. Dataset overview

In [ ]:
print("Shape:", df.shape)
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
display(df.info())
display(df.describe(include="all").T)


## 2. Data quality review

In [ ]:
quality = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str)
})
display(quality)
print("Exact duplicate rows:", df.duplicated().sum())
print("Unique invoices:", df["InvoiceNo"].nunique())
print("Unique known customers:", df["CustomerID"].nunique())
print("Countries:", df["Country"].nunique())


## 3. Define the sales analysis population

In [ ]:
# Exclude cancellations and non-positive line totals from positive-sales revenue analysis.
sales = df[(df["LineTotal"] > 0) & (~df["IsCancellation"].astype(bool))].copy()
sales["YearMonth"] = sales["InvoiceDate"].dt.to_period("M").astype(str)
print("Positive sales rows:", len(sales))
print("Positive sales revenue (£):", round(sales["LineTotal"].sum(), 2))
print("Positive sales invoices:", sales["InvoiceNo"].nunique())


## 4. Descriptive statistics

In [ ]:
display(sales[["Quantity", "UnitPrice", "LineTotal"]].describe().T)
order_revenue = sales.groupby("InvoiceNo")["LineTotal"].sum()
display(order_revenue.describe().to_frame("Order revenue (£)"))


## 5. Monthly revenue trend

In [ ]:
monthly_revenue = sales.groupby(sales["InvoiceDate"].dt.to_period("M"))["LineTotal"].sum()
ax = monthly_revenue.plot(figsize=(11,5), marker="o", title="Monthly Positive Sales Revenue")
ax.set_xlabel("Month"); ax.set_ylabel("Revenue (£)"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
display(monthly_revenue.to_frame("Revenue (£)"))


## 6. Country performance

In [ ]:
country_revenue = sales.groupby("Country")["LineTotal"].sum().sort_values(ascending=False)
display(country_revenue.head(10).to_frame("Revenue (£)"))
country_revenue.head(10).sort_values().plot(kind="barh", figsize=(10,5), title="Top 10 Countries by Revenue")
plt.xlabel("Revenue (£)"); plt.tight_layout(); plt.show()


## 7. Product performance

In [ ]:
product_performance = sales.groupby("Description").agg(
    Quantity_Sold=("Quantity", "sum"),
    Revenue=("LineTotal", "sum")
).sort_values("Revenue", ascending=False)
display(product_performance.head(10))
product_performance.head(10)["Revenue"].sort_values().plot(kind="barh", figsize=(10,5), title="Top 10 Products by Revenue")
plt.xlabel("Revenue (£)"); plt.tight_layout(); plt.show()


## 8. Customer and order behaviour

In [ ]:
customer_revenue = sales.dropna(subset=["CustomerID"]).groupby("CustomerID")["LineTotal"].sum().sort_values(ascending=False)
display(customer_revenue.head(10).to_frame("Revenue (£)"))
print("Rows without CustomerID:", sales["CustomerID"].isna().sum())
print("Share without CustomerID (%):", round(sales["CustomerID"].isna().mean()*100, 2))
order_revenue.clip(upper=order_revenue.quantile(.99)).plot(kind="hist", bins=50, figsize=(10,5),
    title="Order Revenue Distribution (capped at 99th percentile)")
plt.xlabel("Order revenue (£)"); plt.tight_layout(); plt.show()


## 9. Weekday pattern

In [ ]:
weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
weekday_revenue = sales.groupby(sales["InvoiceDate"].dt.day_name())["LineTotal"].sum().reindex(weekday_order)
display(weekday_revenue.to_frame("Revenue (£)"))
weekday_revenue.plot(kind="bar", figsize=(9,5), title="Revenue by Weekday")
plt.xlabel("Weekday"); plt.ylabel("Revenue (£)"); plt.xticks(rotation=30); plt.tight_layout(); plt.show()


## 10. Anomalies and limitations

In [ ]:
print("Negative line totals:", (df["LineTotal"] < 0).sum())
print("Zero line totals:", (df["LineTotal"] == 0).sum())
print("Cancellation-flagged rows:", df["IsCancellation"].astype(bool).sum())
print("Return/negative-quantity-flagged rows:", df["IsReturnOrNegativeQty"].astype(bool).sum())
print("Extreme positive line totals (top 10):")
display(df.nlargest(10, "LineTotal")[["InvoiceNo","StockCode","Description","Quantity","UnitPrice","LineTotal","Country"]])


## 11. Key insights

In [ ]:
uk_share = sales.loc[sales["Country"] == "United Kingdom", "LineTotal"].sum() / sales["LineTotal"].sum() * 100
top5_share = country_revenue.head(5).sum() / sales["LineTotal"].sum() * 100
print(f"1. Positive-value sales revenue totals £{sales['LineTotal'].sum():,.2f} across {sales['InvoiceNo'].nunique():,} invoices.")
print(f"2. United Kingdom contributes {uk_share:.1f}% of positive-value sales revenue.")
print(f"3. Highest monthly revenue: {monthly_revenue.idxmax()} (£{monthly_revenue.max():,.2f}).")
print(f"4. Top five countries contribute {top5_share:.1f}% of positive-value sales revenue.")
print(f"5. Highest-revenue product: {product_performance.index[0]} (£{product_performance.iloc[0]['Revenue']:,.2f}).")
print(f"6. {sales['CustomerID'].isna().mean()*100:.1f}% of positive sales rows lack CustomerID; customer analysis is incomplete.")
print(f"7. Mean order revenue (£{order_revenue.mean():,.2f}) exceeds median (£{order_revenue.median():,.2f}), indicating a right-skewed order distribution.")


## 12. Conclusion

The dataset shows a highly concentrated UK market, strong month-to-month revenue variation, and a long-tailed order-value distribution. Customer-level conclusions must be qualified because many records do not have a CustomerID. December 2011 is only partial in this source extract. The analysis separates positive sales from negative/zero adjustments rather than treating all line totals as sales revenue.

**Suggested next step:** Build a dashboard in Power BI or Excel using the exported monthly, country, and product summary tables.
